In [3]:
#  Check if appearance features cluster by SUBJECT rather than by ENGAGEMENT
from sklearn.metrics import silhouette_score
import numpy as np

# Use test set appearance features (averaged across frames)
test_app_avg = test_features[:, :, :512].mean(axis=1)  # (1784, 512)
test_geo_avg = test_features[:, :, 512:].mean(axis=1)  # (1784, 10)

# How well do features cluster by engagement label?
sil_app = silhouette_score(test_app_avg, test_labels, sample_size=1000, random_state=42)
sil_geo = silhouette_score(test_geo_avg, test_labels, sample_size=1000, random_state=42)

print(f"Silhouette score (appearance vs engagement): {sil_app:.4f}")
print(f"Silhouette score (geometry vs engagement):   {sil_geo:.4f}")
print(f"\nRange: -1 (wrong clusters) to +1 (perfect clusters)")
print(f"Near 0 = no meaningful clustering by engagement state")


Silhouette score (appearance vs engagement): -0.0367
Silhouette score (geometry vs engagement):   -0.0074

Range: -1 (wrong clusters) to +1 (perfect clusters)
Near 0 = no meaningful clustering by engagement state


In [1]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=15)
# Latency: Single-frame methodology (Benyamin, 2026)
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 15                    # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_smote_T15.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T15.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print("ALL EVALUATIONS COMPLETE FOR T = 10")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 15

LOADING DATA
Training:   7851 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 2617, 'Passive': 2617, 'Engaged': 2617}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 246, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 1.0097 | Train F1: 0.5223 | Val F1: 0.3046 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 0.7691 | Train F1: 0.6253 | Val F1: 0.2920 | Best: 0.3046 | Patience: 0/10
Epoch   3/100 | Train Loss: 0.7068 | Train F1: 0.6499 | Val F1: 0.3954 | Best: 0.3046 | Patience: 1/10
Epoch   4/100 | Train Loss: 0.6164 | Train F1: 0.6907 | Val F1: 0.3381 | Best: 0.3954 | Patience: 0/10
Epoch   5/100 | Train Loss: 0.5982 | Train F1: 0.7064 | Val F1: 0.3166 | Best: 0.3954 | Patience: 

I0000 00:00:1792253992.872459       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                  5.78      22.86       3.09     233.23
GhostFaceNet (DeepFace)            126.27      16.78     115.49     226.04
Transformer Classifier               2.36       6.81       1.32      70.05
----------------------------------------------------------------------
TOTAL (single frame)               134.73      28.46     121.23     353.28

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          134.73 ms
  Result:                         FAIL (134.7 ms > 130 ms)

  Benyamin fastest pipeline:      248 m

In [5]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=20)
# Latency: Single-frame methodology (Benyamin, 2026)
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 20                    # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_smote_T20.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T20.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print(f"ALL EVALUATIONS COMPLETE FOR T = {T}")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 20

LOADING DATA
Training:   7851 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 2617, 'Passive': 2617, 'Engaged': 2617}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 246, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 1.2008 | Train F1: 0.3245 | Val F1: 0.0694 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 1.1121 | Train F1: 0.3457 | Val F1: 0.0694 | Best: 0.0694 | Patience: 0/10
Epoch   3/100 | Train Loss: 1.0376 | Train F1: 0.4198 | Val F1: 0.2495 | Best: 0.0694 | Patience: 1/10
Epoch   4/100 | Train Loss: 0.9144 | Train F1: 0.5122 | Val F1: 0.2421 | Best: 0.2495 | Patience: 0/10
Epoch   5/100 | Train Loss: 0.8485 | Train F1: 0.5600 | Val F1: 0.2833 | Best: 0.2495 | Patience: 

I0000 00:00:1792254413.554426       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                  3.68       1.18       3.14      10.26
GhostFaceNet (DeepFace)            126.74      15.74     112.00     216.24
Transformer Classifier               1.76       0.33       1.41       3.83
----------------------------------------------------------------------
TOTAL (single frame)               132.49      16.63     117.18     228.91

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          132.49 ms
  Result:                         FAIL (132.5 ms > 130 ms)

  Benyamin fastest pipeline:      248 m

In [1]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=30)
# Latency: Single-frame methodology (Benyamin, 2026)
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 30                    # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_smote_T30.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T30.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print(f"ALL EVALUATIONS COMPLETE FOR T = {T}")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 30

LOADING DATA
Training:   7851 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 2617, 'Passive': 2617, 'Engaged': 2617}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 246, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 1.1132 | Train F1: 0.4215 | Val F1: 0.3348 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 0.8564 | Train F1: 0.5679 | Val F1: 0.2827 | Best: 0.3348 | Patience: 0/10
Epoch   3/100 | Train Loss: 0.7380 | Train F1: 0.6425 | Val F1: 0.2031 | Best: 0.3348 | Patience: 1/10
Epoch   4/100 | Train Loss: 0.6765 | Train F1: 0.6686 | Val F1: 0.3533 | Best: 0.3348 | Patience: 2/10
Epoch   5/100 | Train Loss: 0.6220 | Train F1: 0.6950 | Val F1: 0.3687 | Best: 0.3533 | Patience: 

I0000 00:00:1792376341.831119       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                 11.73      44.09       3.55     448.86
GhostFaceNet (DeepFace)            222.54      68.70     146.25     684.22
Transformer Classifier               3.41       2.19       1.61      15.70
----------------------------------------------------------------------
TOTAL (single frame)               238.35      87.90     152.72     761.82

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          238.35 ms
  Result:                         FAIL (238.4 ms > 130 ms)

  Benyamin fastest pipeline:      248 m

In [5]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=30)
# Latency: Single-frame methodology (Benyamin, 2026)
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 5                   # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_smote_T05.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T05.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print(f"ALL EVALUATIONS COMPLETE FOR T = {T}")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 5

LOADING DATA
Training:   7851 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 2617, 'Passive': 2617, 'Engaged': 2617}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 246, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 1.0274 | Train F1: 0.4956 | Val F1: 0.2811 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 0.8036 | Train F1: 0.6066 | Val F1: 0.3684 | Best: 0.2811 | Patience: 0/10
Epoch   3/100 | Train Loss: 0.7250 | Train F1: 0.6405 | Val F1: 0.2125 | Best: 0.3684 | Patience: 0/10
Epoch   4/100 | Train Loss: 0.6578 | Train F1: 0.6843 | Val F1: 0.2579 | Best: 0.3684 | Patience: 1/10
Epoch   5/100 | Train Loss: 0.6189 | Train F1: 0.7010 | Val F1: 0.3762 | Best: 0.3684 | Patience: 2

I0000 00:00:1792427216.904496       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                  3.33       0.18       3.12       3.88
GhostFaceNet (DeepFace)            115.21       2.01     113.50     126.60
Transformer Classifier               1.77       0.12       1.40       2.14
----------------------------------------------------------------------
TOTAL (single frame)               120.63       2.03     118.78     132.55

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          120.63 ms
  Result:                         PASS (120.6 ms <= 130 ms)

  Benyamin fastest pipeline:      248 

In [7]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=30)
# Latency: Single-frame methodology (Benyamin, 2026)
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 10                    # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_smote_T10.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T10.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print(f"ALL EVALUATIONS COMPLETE FOR T = {T}")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 10

LOADING DATA
Training:   7851 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 2617, 'Passive': 2617, 'Engaged': 2617}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 246, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 1.0164 | Train F1: 0.5086 | Val F1: 0.2476 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 0.7908 | Train F1: 0.6047 | Val F1: 0.3357 | Best: 0.2476 | Patience: 0/10
Epoch   3/100 | Train Loss: 0.7227 | Train F1: 0.6481 | Val F1: 0.3450 | Best: 0.3357 | Patience: 0/10
Epoch   4/100 | Train Loss: 0.6548 | Train F1: 0.6758 | Val F1: 0.3820 | Best: 0.3450 | Patience: 0/10
Epoch   5/100 | Train Loss: 0.5968 | Train F1: 0.7054 | Val F1: 0.3679 | Best: 0.3820 | Patience: 

I0000 00:00:1792428102.518347       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                  6.22      29.31       3.04     297.86
GhostFaceNet (DeepFace)            111.37       2.24     109.01     126.06
Transformer Classifier               1.71       0.10       1.38       2.04
----------------------------------------------------------------------
TOTAL (single frame)               119.60      29.41     114.21     411.35

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          119.60 ms
  Result:                         PASS (119.6 ms <= 130 ms)

  Benyamin fastest pipeline:      248 

In [9]:
# ============================================================
# COMPLETE TRAINING, EVALUATION & LATENCY BENCHMARK
# Model: Hybrid Engagement Transformer (T=30)
# Use No-SMOTE training for comparison
# ============================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import time
import cv2
from deepface import DeepFace
from feature_extractor import EngagementFeatureExtractor

# ============================================================
# 1. CONFIGURATION
# ============================================================

T = 30                    # Sequence length (frames per video)
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4       # L2 regularization
PATIENCE = 10             # Early stopping patience
MAX_EPOCHS = 100
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 2
NUM_CLASSES = 3
DIM_APPEARANCE = 512
DIM_GEOMETRY = 10
DIM_INPUT = DIM_APPEARANCE + DIM_GEOMETRY  # 522

LABELS_MAP = {0: "Disengaged", 1: "Passive", 2: "Engaged"}

# Paths
TRAIN_PATH = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_train_master_T30.npz")
VAL_PATH   = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_val_master_T30.npz")
TEST_PATH  = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/features_test_master_T30.npz")
MODEL_SAVE = Path("/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T30.pt")
TEST_CSV   = Path("/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels_TriState.csv")
TEST_VIDEO = Path("/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"Sequence length T = {T}")

# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# --- Training data (SMOTE-balanced, already T=10) ---
train_data = np.load(TRAIN_PATH)
train_app = train_data["appearance"]   # (7851, 10, 512)
train_geo = train_data["geometry"]     # (7851, 10, 10)
train_labels = train_data["labels"]    # (7851,)
train_features = np.concatenate([train_app, train_geo], axis=-1)  # (7851, 10, 522)
print(f"Training:   {train_features.shape[0]} samples (SMOTE-balanced)")

# --- Validation data (no SMOTE, subsample to T) ---
val_data = np.load(VAL_PATH)
val_indices = np.linspace(0, 29, T, dtype=int)
val_app = val_data["appearance"][:, val_indices, :]
val_geo = val_data["geometry"][:, val_indices, :]
val_labels = val_data["labels"]
val_features = np.concatenate([val_app, val_geo], axis=-1)
print(f"Validation: {val_features.shape[0]} samples (original distribution)")

# --- Test data (no SMOTE, subsample to T) ---
test_data = np.load(TEST_PATH)
test_indices = np.linspace(0, 29, T, dtype=int)
test_app = test_data["appearance"][:, test_indices, :]
test_geo = test_data["geometry"][:, test_indices, :]
test_labels = test_data["labels"]
test_features = np.concatenate([test_app, test_geo], axis=-1)
print(f"Test:       {test_features.shape[0]} samples (original distribution)")

# --- Class distributions ---
for name, labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = {LABELS_MAP[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {name} distribution: {dist}")

# --- Create DataLoaders ---
train_dataset = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(val_features, dtype=torch.float32),
    torch.tensor(val_labels, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# ============================================================
# 3. MODEL DEFINITION
# ============================================================

class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        self.relu = nn.ReLU()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=NUM_LAYERS
        )
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.relu(self.fusion_proj(x))
        x = self.transformer(x)
        x = x.mean(dim=1)  # Mean pooling across time
        return self.classifier(x)

model = HybridEngagementModel().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

# ============================================================
# 4. TRAINING WITH EARLY STOPPING
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_preds = []
    train_true = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    train_loss /= len(train_dataset)
    train_f1 = f1_score(train_true, train_preds, average="macro")

    # --- Validation phase (no weight updates) ---
    model.eval()
    val_preds = []
    val_true = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(batch_y.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")

    # --- Print epoch summary ---
    print(f"Epoch {epoch:>3}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | "
          f"Val F1: {val_f1:.4f} | "
          f"Best: {best_val_f1:.4f} | Patience: {patience_counter}/{PATIENCE}")

    # --- Early stopping check ---
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}. Best Val F1 = {best_val_f1:.4f} (epoch {best_epoch})")
            break

print(f"\nBest Validation F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE}")

# ============================================================
# 5. TEST EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=device))
model.eval()

test_preds = []
test_true = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        test_true.extend(batch_y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, average="macro")
test_acc = accuracy_score(test_true, test_preds)

print(f"\nMacro F1-Score: {test_f1:.4f}")
print(f"Accuracy:       {test_acc:.4f}")

print(f"\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=["Disengaged", "Passive", "Engaged"]))

print("Confusion Matrix:")
cm = confusion_matrix(test_true, test_preds)
print(f"{'':>15} {'Pred Dis':>10} {'Pred Pas':>10} {'Pred Eng':>10}")
for i, row_label in enumerate(["Actual Dis", "Actual Pas", "Actual Eng"]):
    print(f"{row_label:>15} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ============================================================
# 6. MODEL-ONLY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("MODEL-ONLY LATENCY (Transformer forward pass)")
print("=" * 60)

dummy = torch.randn(1, T, DIM_INPUT).to(device)

# Warm-up
for _ in range(20):
    with torch.no_grad():
        _ = model(dummy)

model_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    model_times.append((time.perf_counter() - start) * 1000)

model_times = np.array(model_times)
print(f"  Mean: {model_times.mean():.2f} ms")
print(f"  Std:  {model_times.std():.2f} ms")
print(f"  Min:  {model_times.min():.2f} ms")
print(f"  Max:  {model_times.max():.2f} ms")

# ============================================================
# 7. SINGLE-FRAME LATENCY (Benyamin 2026 Methodology)
# ============================================================

print("\n" + "=" * 60)
print("SINGLE-FRAME AI PROCESSING TIME")
print("Methodology: Benyamin (2026) Latency-Wall Framework")
print("=" * 60)

# --- Initialize feature extractor ---
extractor = EngagementFeatureExtractor()

# --- Load a representative test video frame ---
test_df = pd.read_csv(TEST_CSV)
test_video_path = None
for idx in range(len(test_df)):
    clip_id = test_df["ClipID"].iloc[idx].strip()
    stem = Path(clip_id).stem
    subject_folder = stem[:6]
    vpath = TEST_VIDEO / subject_folder / stem / clip_id
    if vpath.exists():
        test_video_path = str(vpath)
        break

assert test_video_path is not None, "No test video found"

cap = cv2.VideoCapture(test_video_path)
ret, frame = cap.read()
cap.release()
assert ret, "Failed to read frame"
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
print(f"\nTest video: {test_video_path}")
print(f"Frame shape: {frame.shape}")

# --- Warm-up (10 runs) ---
print("Running 10 warm-up passes...")
for _ in range(10):
    _ = extractor.face_mesh.process(frame_rgb)
    _ = DeepFace.represent(img_path=frame, model_name="GhostFaceNet",
                           enforce_detection=False, align=True)
    dummy_in = torch.randn(1, 1, DIM_INPUT).to(device)
    with torch.no_grad():
        _ = model(dummy_in)

# --- Timed runs (100 iterations) ---
NUM_RUNS = 100
t_mediapipe = []
t_ghostface = []
t_transformer = []
t_total = []

print(f"Running {NUM_RUNS} timed single-frame benchmarks...\n")

for i in range(NUM_RUNS):
    total_start = time.perf_counter()

    # Stage 1: MediaPipe Face Mesh
    mp_start = time.perf_counter()
    results_mp = extractor.face_mesh.process(frame_rgb)
    if results_mp.multi_face_landmarks:
        landmarks = results_mp.multi_face_landmarks[0].landmark
        geo_feat = extractor.get_geometric_features(landmarks, frame.shape)
    else:
        geo_feat = [0.0] * DIM_GEOMETRY
    mp_end = time.perf_counter()

    # Stage 2: GhostFaceNet
    gf_start = time.perf_counter()
    try:
        embedding_obj = DeepFace.represent(
            img_path=frame, model_name="GhostFaceNet",
            enforce_detection=False, align=True
        )
        app_feat = embedding_obj[0]["embedding"]
    except:
        app_feat = [0.0] * DIM_APPEARANCE
    gf_end = time.perf_counter()

    # Stage 3: Transformer classification
    combined = np.concatenate([app_feat, geo_feat])
    tensor_in = torch.tensor(combined, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tf_start = time.perf_counter()
    with torch.no_grad():
        logits = model(tensor_in)
        pred = torch.argmax(logits, dim=1).item()
    tf_end = time.perf_counter()

    total_end = time.perf_counter()

    t_mediapipe.append((mp_end - mp_start) * 1000)
    t_ghostface.append((gf_end - gf_start) * 1000)
    t_transformer.append((tf_end - tf_start) * 1000)
    t_total.append((total_end - total_start) * 1000)

t_mediapipe = np.array(t_mediapipe)
t_ghostface = np.array(t_ghostface)
t_transformer = np.array(t_transformer)
t_total = np.array(t_total)

# --- Results ---
print("=" * 60)
print("SINGLE-FRAME LATENCY RESULTS")
print("=" * 60)
print(f"\n{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 70)
print(f"{'MediaPipe Face Mesh':<30} {t_mediapipe.mean():>10.2f} {t_mediapipe.std():>10.2f} {t_mediapipe.min():>10.2f} {t_mediapipe.max():>10.2f}")
print(f"{'GhostFaceNet (DeepFace)':<30} {t_ghostface.mean():>10.2f} {t_ghostface.std():>10.2f} {t_ghostface.min():>10.2f} {t_ghostface.max():>10.2f}")
print(f"{'Transformer Classifier':<30} {t_transformer.mean():>10.2f} {t_transformer.std():>10.2f} {t_transformer.min():>10.2f} {t_transformer.max():>10.2f}")
print("-" * 70)
print(f"{'TOTAL (single frame)':<30} {t_total.mean():>10.2f} {t_total.std():>10.2f} {t_total.min():>10.2f} {t_total.max():>10.2f}")

# --- vs 130 ms Latency Wall ---
print(f"\n{'=' * 60}")
print("LATENCY-WALL COMPARISON (Benyamin, 2026)")
print(f"{'=' * 60}")
print(f"  AI Latency Budget:              130 ms")
print(f"  My Single-Frame Total:          {t_total.mean():.2f} ms")
if t_total.mean() <= 130:
    print(f"  Result:                         PASS ({t_total.mean():.1f} ms <= 130 ms)")
else:
    print(f"  Result:                         FAIL ({t_total.mean():.1f} ms > 130 ms)")

# --- vs Benyamin's fastest pipeline ---
benyamin_total = 248
print(f"\n  Benyamin fastest pipeline:      {benyamin_total} ms (YOLOv11n + ViT-FER)")
print(f"  My pipeline:                    {t_total.mean():.2f} ms")
ratio = benyamin_total / t_total.mean()
print(f"  Speedup:                        {ratio:.1f}x {'faster' if ratio > 1 else 'slower'}")

# --- Component breakdown ---
print(f"\n{'=' * 60}")
print("COMPONENT BREAKDOWN")
print(f"{'=' * 60}")
total_mean = t_total.mean()
print(f"  MediaPipe Face Mesh:            {t_mediapipe.mean():>7.2f} ms  ({t_mediapipe.mean()/total_mean*100:>5.1f}%)")
print(f"  GhostFaceNet:                   {t_ghostface.mean():>7.2f} ms  ({t_ghostface.mean()/total_mean*100:>5.1f}%)")
print(f"  Transformer Classifier:         {t_transformer.mean():>7.2f} ms  ({t_transformer.mean()/total_mean*100:>5.1f}%)")

# --- Side-by-side table ---
print(f"\n{'=' * 60}")
print("SIDE-BY-SIDE: MY PIPELINE vs BENYAMIN (2026)")
print(f"{'=' * 60}")
print(f"{'Stage':<28} {'Benyamin':>12} {'My Praxis':>12}")
print(f"{'-' * 52}")
print(f"{'Face Detection + Geometry':<28} {'54 ms':>12} {f'{t_mediapipe.mean():.1f} ms':>12}")
print(f"{'Embedding + Classification':<28} {'194 ms':>12} {f'{(t_ghostface.mean()+t_transformer.mean()):.1f} ms':>12}")
print(f"{'-' * 52}")
print(f"{'TOTAL':<28} {'248 ms':>12} {f'{t_total.mean():.1f} ms':>12}")
print(f"{'vs 130 ms budget':<28} {'FAIL':>12} {'PASS' if t_total.mean() <= 130 else 'FAIL':>12}")

# --- Extrapolated multi-frame (reference) ---
print(f"\n{'=' * 60}")
print("REFERENCE: EXTRAPOLATED SEQUENTIAL MULTI-FRAME LATENCY")
print(f"{'=' * 60}")
for T_ref in [5, 10, 15, 20]:
    multi = t_total.mean() * T_ref
    print(f"  T={T_ref:>2} frames (sequential): {multi:>8.1f} ms  [streaming architecture eliminates this]")

print(f"\n  Last prediction: {LABELS_MAP[pred]}")

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print(f"{'=' * 60}")
print(f"  Model:                  Hybrid Engagement Transformer")
print(f"  Sequence Length:        T = {T}")
print(f"  Parameters:             {total_params:,}")
print(f"  Training Epochs:        {best_epoch} (early stopped)")
print(f"  Best Validation F1:     {best_val_f1:.4f}")
print(f"  Test Macro F1:          {test_f1:.4f}")
print(f"  Test Accuracy:          {test_acc:.4f}")
print(f"  Model-Only Latency:     {model_times.mean():.2f} ms")
print(f"  Single-Frame Latency:   {t_total.mean():.2f} ms")
print(f"  Latency Wall (130 ms):  {'PASS' if t_total.mean() <= 130 else 'FAIL'}")
print(f"  vs Benyamin (248 ms):   {ratio:.1f}x faster")
print(f"{'=' * 60}")
print(f"ALL EVALUATIONS COMPLETE FOR T = {T}")
print(f"{'=' * 60}")

Device: mps
Sequence length T = 30

LOADING DATA
Training:   5358 samples (SMOTE-balanced)
Validation: 1429 samples (original distribution)
Test:       1784 samples (original distribution)
  Train distribution: {'Disengaged': 247, 'Passive': 2617, 'Engaged': 2494}
  Val distribution: {'Disengaged': 166, 'Passive': 813, 'Engaged': 450}
  Test distribution: {'Disengaged': 88, 'Passive': 882, 'Engaged': 814}

Batches — Train: 168, Val: 45, Test: 56

Model parameters: 6,574,083

TRAINING
Epoch   1/100 | Train Loss: 0.9532 | Train F1: 0.3221 | Val F1: 0.1597 | Best: 0.0000 | Patience: 0/10
Epoch   2/100 | Train Loss: 0.8635 | Train F1: 0.3320 | Val F1: 0.3272 | Best: 0.1597 | Patience: 0/10
Epoch   3/100 | Train Loss: 0.8264 | Train F1: 0.3930 | Val F1: 0.3343 | Best: 0.3272 | Patience: 0/10
Epoch   4/100 | Train Loss: 0.8097 | Train F1: 0.4142 | Val F1: 0.3290 | Best: 0.3343 | Patience: 0/10
Epoch   5/100 | Train Loss: 0.8310 | Train F1: 0.3722 | Val F1: 0.3412 | Best: 0.3343 | Patience: 1

I0000 00:00:1792440667.100127       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


✅ GhostFaceNet Loaded (512-Dim)

Test video: /Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test/500044/5000441001/5000441001.avi
Frame shape: (480, 640, 3)
Running 10 warm-up passes...
Running 100 timed single-frame benchmarks...

SINGLE-FRAME LATENCY RESULTS

Component                       Mean (ms)   Std (ms)   Min (ms)   Max (ms)
----------------------------------------------------------------------
MediaPipe Face Mesh                  3.33       0.34       3.10       4.75
GhostFaceNet (DeepFace)            118.44      10.11     108.34     165.86
Transformer Classifier               1.74       0.24       1.43       3.50
----------------------------------------------------------------------
TOTAL (single frame)               123.82      10.14     113.75     170.82

LATENCY-WALL COMPARISON (Benyamin, 2026)
  AI Latency Budget:              130 ms
  My Single-Frame Total:          123.82 ms
  Result:                         PASS (123.8 ms <= 130 ms)

  Benyamin fastest pipeline:      248 